In [ ]:
from vnstock import Listing, Quote

c:\Users\BAORTG\AppData\Local\Programs\Python\Python312\Lib\site-packages\vnai\scope\profile.py:562: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Phiên bản Vnstock 3.5.0 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnstock --upgrade`.
Lịch sử phiên bản: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban
Phiên bản hiện tại 3.2.6

Phiên bản Vnai 2.4.0 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnai --upgrade`.
Lịch sử phiên bản: https://pypi.org/project/vnai/#history
Phiên bản hiện tại 2.1.9

## 1. List of symbols by HOSE

In [2]:
vn30_symbols = ['FPT', 'VIC', 'GAS', 'SSI', 'HPG', 'VNM', 'VCB', 'STB', 'MSN', 'MWG']
vn30_vnsi_symbols = ['MWG', 'BID', 'VCB', 'VIC', 'MBB']

In [3]:
len(vn30_symbols)
len(vn30_vnsi_symbols)

5

## 2. Getting Quote

#### &emsp;&emsp;Config

In [4]:
"""TRAIN_START_DATE = '2022-01-01'
TRAIN_END_DATE = '2023-12-31'

VALIDATE_START_DATE = '2024-01-01'
VALIDATE_END_DATE = '2024-06-30'

TEST_START_DATE = '2024-07-01'
TEST_END_DATE = '2025-06-30'"""

START_DATE = '2015-01-01'
END_DATE = '2025-12-31'


folder_path = '../dataset'

In [5]:
def process_data(symbol, start, end):
    try:
        quote = Quote(symbol=symbol, source='VCI')
        quote_df = quote.history(start=start, end=end, interval='1D')
        quote_df['tic'] = symbol
        quote_df['day'] = quote_df['time'].dt.dayofweek
        return quote_df
    except Exception as e:
        print(f"Lỗi khi xử lý mã {symbol}: {e}")
        return None

In [6]:
def chunk_list(lst, size):
    """Chia list thành các đoạn nhỏ"""
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [7]:
def init_data_set(list_of_symbols, start, end, folder_path, title):

    file_path = os.path.join(folder_path, f'{title}_data.xlsx')

    skipped_symbols = []

    # Nếu file chưa tồn tại, tạo mới
    if not os.path.exists(file_path):
        print("File chưa tồn tại, đang tạo mới...")
        # Tạo file rỗng
        df_empty = pd.DataFrame(columns=['date', 'open', 'high', 'low', 'close', 'volume', 'tic', 'day'])
        df_empty.to_excel(file_path, index=False)
    
    print("Tiến hành ghi file")
    # Get sheet name
    # Xác định sheet cần ghi
    sheet_name = load_workbook(file_path).sheetnames[0]

    #for batch_idx, batch in enumerate(chunk_list(list_of_symbols, 30)):  # mỗi batch 30 mã
            #print(f"Đang xử lý batch {batch_idx + 1} / {(len(list_of_symbols)//30)+1}")
        
    for symbol in list_of_symbols:

        print(f'Đang xử lý cổ phiếu {symbol}')

        # Load data
        df = process_data(symbol, start, end)

        if df is None or df.empty:
            print(f'skip symbol {symbol}')
            skipped_symbols.append(symbol)
            continue

        # Find where the last row is
        wb = load_workbook(file_path)
        start_row = wb[sheet_name].max_row
        wb.close()

        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:    
            df.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=start_row)

            #print(f'completed batch {batch_idx + 1}')

            # nghỉ giữa các batch để tránh bị giới hạn request
            #time.sleep(10)



In [8]:
symbol_batches = list(chunk_list(vn30_vnsi_symbols, 2))
len(symbol_batches)

3

## Writing Data

In [9]:
init_data_set(vn30_vnsi_symbols, START_DATE, END_DATE, folder_path, title='5_vn30_vnsi_symbols')

File chưa tồn tại, đang tạo mới...
Tiến hành ghi file
Đang xử lý cổ phiếu MWG
Đang xử lý cổ phiếu BID
Đang xử lý cổ phiếu VCB
Đang xử lý cổ phiếu VIC
Đang xử lý cổ phiếu MBB
